# EDA: Player Stats


## Purpose
Distribution of VORP, BPM, WS/48. How much star power is needed to win a championship? Which player metrics best separate series winners from losers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
from pathlib import Path
import sys; sys.path.insert(0, str(Path(".").parent / "src"))
from nba_predictor.config import cfg


## Load Player Features

In [ ]:
player_path = cfg.project_root / "data/processed/player_season_features.parquet"
if player_path.exists():
    df = pd.read_parquet(player_path)
    print(df.shape)
else:
    print("Run make process first")


## VORP and BPM Distributions

In [ ]:
Path("../reports/figures").mkdir(parents=True, exist_ok=True)

series_path = cfg.project_root / "data/processed/series_dataset.parquet"
series_df = pd.read_parquet(series_path)

# Identify playoff teams by season
playoff_teams = set()
for _, row in series_df.iterrows():
    playoff_teams.add((row["season"], row["higher_seed"]))
    playoff_teams.add((row["season"], row["lower_seed"]))

df["is_playoff"] = df.apply(lambda r: (r["season"], r["Team_abbrev"]) in playoff_teams, axis=1)

# Champions
finals = series_df[series_df["series_round"] == "Finals"]
champions = {(row["season"], row["series_winner"]) for _, row in finals.iterrows()}
df["is_champion"] = df.apply(lambda r: (r["season"], r["Team_abbrev"]) in champions, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# VORP sum: playoff vs non-playoff
ax = axes[0]
ax.hist(df[~df["is_playoff"]]["team_VORP_sum"].dropna(), bins=25, alpha=0.5,
        color="steelblue", label="Non-playoff", density=True)
ax.hist(df[df["is_playoff"]]["team_VORP_sum"].dropna(), bins=25, alpha=0.6,
        color="gold", label="Playoff", density=True)
ax.hist(df[df["is_champion"]]["team_VORP_sum"].dropna(), bins=15, alpha=0.8,
        color="crimson", label="Champion", density=True)
ax.set_xlabel("Team VORP Sum"); ax.set_ylabel("Density")
ax.set_title("Team VORP Sum: Playoff vs Non-Playoff vs Champion")
ax.legend()

# Star BPM: playoff vs champion
ax2 = axes[1]
ax2.hist(df[df["is_playoff"] & ~df["is_champion"]]["Star_player_BPM"].dropna(),
         bins=25, alpha=0.5, color="steelblue", label="Playoff (non-champ)", density=True)
ax2.hist(df[df["is_champion"]]["Star_player_BPM"].dropna(),
         bins=15, alpha=0.8, color="crimson", label="Champion", density=True)
ax2.axvline(3.0, color="orange", ls="--", lw=1.5, label="Star threshold (BPM=3)")
ax2.axvline(5.0, color="red", ls="--", lw=1.5, label="All-NBA threshold (BPM=5)")
ax2.set_xlabel("Star Player BPM"); ax2.set_ylabel("Density")
ax2.set_title("Star Player BPM: Champions vs Playoff teams")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig("../reports/figures/02_vorp_bpm_distributions.png", dpi=120, bbox_inches="tight")
plt.show()

print("Median team VORP sum:")
print(f"  Non-playoff: {df[~df['is_playoff']]['team_VORP_sum'].median():.2f}")
print(f"  Playoff:     {df[df['is_playoff']]['team_VORP_sum'].median():.2f}")
print(f"  Champion:    {df[df['is_champion']]['team_VORP_sum'].median():.2f}")


## Star Power Required to Win Championships

In [ ]:
# Champion star power by season
champ_rows = df[df["is_champion"]].sort_values("season")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
seasons = champ_rows["season"]
ax.bar(seasons, champ_rows["Top3_VORP_sum"], color="gold", edgecolor="black", linewidth=0.5)
ax.axhline(champ_rows["Top3_VORP_sum"].mean(), color="red", ls="--", lw=1.5,
           label=f"Mean = {champ_rows['Top3_VORP_sum'].mean():.1f}")
ax.set_xlabel("Season"); ax.set_ylabel("Top-3 Player VORP Sum")
ax.set_title("Champion Top-3 VORP Sum by Season")
ax.legend(); ax.grid(alpha=0.3, axis="y")

ax2 = axes[1]
ax2.scatter(champ_rows["Star_player_BPM"], champ_rows["Top3_VORP_sum"],
            c=champ_rows["season"], cmap="viridis", s=80, edgecolors="black", zorder=3)
ax2.axvline(5.0, color="red", ls="--", lw=1, label="All-NBA BPM threshold (5.0)")
ax2.axhline(champ_rows["Top3_VORP_sum"].mean(), color="gray", ls="--", lw=1)
for _, row in champ_rows.iterrows():
    ax2.annotate(str(int(row["season"]))[-2:], (row["Star_player_BPM"], row["Top3_VORP_sum"]),
                 textcoords="offset points", xytext=(4, 2), fontsize=7)
sm = plt.cm.ScalarMappable(cmap="viridis",
     norm=plt.Normalize(champ_rows["season"].min(), champ_rows["season"].max()))
plt.colorbar(sm, ax=ax2, label="Season")
ax2.set_xlabel("Star Player BPM"); ax2.set_ylabel("Top-3 VORP Sum")
ax2.set_title("Champion Star BPM vs Top-3 VORP")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig("../reports/figures/02_champion_star_power.png", dpi=120, bbox_inches="tight")
plt.show()

# What % of champions had an All-NBA caliber star?
allnba_champs = (champ_rows["Star_player_BPM"] >= 5.0).mean()
print(f"Champions with All-NBA star (BPM ≥ 5): {allnba_champs:.0%}")
print(f"Champions with star player (BPM ≥ 3):  {(champ_rows['Star_player_BPM'] >= 3.0).mean():.0%}")


## Player Momentum Analysis

In [ ]:
# Player momentum features are derived from per-game logs and stored in series_dataset
# Use series-level delta_Star_BPM and delta_VORP as proxies for momentum advantage

series_path = cfg.project_root / "data/processed/series_dataset.parquet"
sf = pd.read_parquet(series_path)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col, label in zip(axes,
        ["delta_VORP", "delta_BPM", "delta_Star_BPM"],
        ["Delta VORP (higher − lower seed)", "Delta BPM", "Delta Star BPM"]):
    winners  = sf[sf["higher_seed_wins"] == 1][col].dropna()
    losers   = sf[sf["higher_seed_wins"] == 0][col].dropna()
    ax.hist(losers,  bins=25, alpha=0.5, color="tomato",    label="Upset (lower seed wins)", density=True)
    ax.hist(winners, bins=25, alpha=0.5, color="steelblue", label="Favorite wins",           density=True)
    ax.axvline(0, color="black", lw=1)
    ax.axvline(winners.mean(), color="steelblue", ls="--", lw=1.5)
    ax.axvline(losers.mean(),  color="tomato",    ls="--", lw=1.5)
    ax.set_xlabel(label); ax.set_ylabel("Density")
    ax.set_title(f"{col}\n(higher seed = positive)")
    ax.legend(fontsize=8)

plt.suptitle("Player Advantage Distributions: Favorites vs. Upsets", fontsize=13)
plt.tight_layout()
plt.savefig("../reports/figures/02_player_momentum_analysis.png", dpi=120, bbox_inches="tight")
plt.show()

# Point-biserial correlation with series outcome
from scipy import stats
for col in ["delta_VORP", "delta_BPM", "delta_Star_BPM", "delta_Top3_VORP"]:
    valid = sf[[col, "higher_seed_wins"]].dropna()
    r, p = stats.pointbiserialr(valid["higher_seed_wins"], valid[col])
    print(f"  {col:25s}  r={r:+.3f}  p={p:.4f}  {'*' if p < 0.05 else ''}")
